# Check AORC mean areal values

Checks the mean areal values written by `calculate-aorc-mean-areal-values` against the source:

1. **Sanity checks** on everything written for the polygon layer: counts, duplicates, ranges, units, times.
2. **Spot check** of a few polygons and hours against an independent zonal mean, computed with `exactextract` directly on the AORC grid. The flow weights each pixel by its coverage fraction times its area (cos(latitude) on a lat/lon grid), so the check uses exactextract's `weighted_mean` with a cos(latitude) weights raster; values should match to float32 precision.
3. **Rerun check**: snapshot the values, rerun the flow (e.g. local mode vs executors), and compare.

Keep `HOURS` and `SAMPLE_IDS` small: each hour is read from NOAA's S3 bucket.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import rioxarray  # noqa: F401
import xarray as xr
from exactextract import exact_extract
from obstore.store import S3Store
from zarr.storage import ObjectStore

from teehr import RemoteReadWriteEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session

#### Settings: match the flow run's arguments

In [ ]:
LOCATION_ID_PREFIX = "usgsbasin"  # the flow's location_id_prefix
START, END = "2020-01-01 00:00", "2020-01-03 23:00"  # the flow's start_dt/end_dt (UTC)
HOURS = None  # e.g. pd.date_range("2020-01-02 15:00", periods=3, freq="h"); None takes the 3 wettest hours
SAMPLE_IDS = None  # e.g. ["usgsbasin-01013500"]; None takes the 3 wettest polygons in those hours

CONFIGURATION_NAME = "aorc_v1_1"
VARIABLE_NAME = "rainrate_hourly_mean"
GRID_VARIABLE_NAME = "APCP_surface"
VALUE_SCALE = 1 / 3600  # kg/m^2 per hour -> mm/s, as the flow applies
TABLE = "primary_timeseries"

In [ ]:
spark = create_spark_session()
ev = RemoteReadWriteEvaluation(spark=spark)
catalog = ev.active_catalog
table = f"{catalog.catalog_name}.{catalog.namespace_name}.{TABLE}"
where = (
    f"configuration_name = '{CONFIGURATION_NAME}' AND variable_name = '{VARIABLE_NAME}' "
    f"AND location_id LIKE '{LOCATION_ID_PREFIX}-%' "
    f"AND value_time BETWEEN TIMESTAMP '{START}' AND TIMESTAMP '{END}'"
)

## 1. Sanity checks

- `rows` should be about `locations × hours` (fewer only where a polygon lacked coverage).
- `unit_name` should be `mm/s`; precipitation never negative and rarely above ~0.03 mm/s (~100 mm/h).
- `value_time` should cover `START..END` hourly, in UTC.

In [ ]:
spark.sql(f"""
    SELECT
        count(*) AS rows,
        count(DISTINCT location_id) AS locations,
        count(DISTINCT value_time) AS hours,
        count(DISTINCT location_id) * count(DISTINCT value_time) AS expected_rows,
        min(value_time) AS first_time,
        max(value_time) AS last_time,
        min(value) AS min_value,
        max(value) AS max_value,
        sum(CASE WHEN value < 0 THEN 1 ELSE 0 END) AS negative,
        sum(CASE WHEN isnan(value) THEN 1 ELSE 0 END) AS nan,
        collect_set(unit_name) AS units,
        collect_set(reference_time) AS reference_times
    FROM {table}
    WHERE {where}
""").toPandas().T

Duplicates: should be empty.

In [ ]:
spark.sql(f"""
    SELECT location_id, value_time, count(*) AS n
    FROM {table}
    WHERE {where}
    GROUP BY location_id, value_time
    HAVING count(*) > 1
    LIMIT 20
""").toPandas()

Missing hours: polygon-hours with no row. Should be empty unless a polygon lacks coverage.

In [ ]:
spark.sql(f"""
    WITH locs AS (SELECT DISTINCT location_id FROM {table} WHERE {where}),
    hours AS (
        SELECT explode(sequence(TIMESTAMP '{START}', TIMESTAMP '{END}', INTERVAL 1 HOUR)) AS value_time
    )
    SELECT l.location_id, h.value_time
    FROM locs l CROSS JOIN hours h
    LEFT ANTI JOIN (SELECT location_id, value_time FROM {table} WHERE {where}) t
        ON t.location_id = l.location_id AND t.value_time = h.value_time
    LIMIT 20
""").toPandas()

## 2. Spot check against exactextract

Opens AORC as the flow does, reads only the sample polygons' window for `HOURS`, and computes each polygon's
coverage- and area-weighted mean with exactextract. A second, rougher check averages the pixels whose centers fall
inside each polygon; that should be close (within a few percent for large polygons) but not equal.
By default the sample is the wettest hours and polygons, since zeros match trivially (and polygons outside AORC have
no rows).

In [ ]:
polygons = ev.locations.filter(
    filters=[{"column": "id", "operator": "like", "value": f"{LOCATION_ID_PREFIX}-%"}]
).to_geopandas()
# Zeros match trivially, so by default sample where it rained
if HOURS is None:
    HOURS = pd.DatetimeIndex(spark.sql(
        f"SELECT value_time FROM {table} WHERE {where} GROUP BY value_time ORDER BY sum(value) DESC LIMIT 3"
    ).toPandas()["value_time"]).sort_values()
if SAMPLE_IDS is None:
    hours_sql = ", ".join(f"TIMESTAMP '{t}'" for t in HOURS)
    SAMPLE_IDS = spark.sql(
        f"SELECT location_id FROM {table} WHERE {where} AND value_time IN ({hours_sql}) "
        "GROUP BY location_id ORDER BY sum(value) DESC LIMIT 3"
    ).toPandas()["location_id"].tolist()
polygons = polygons[polygons["id"].isin(SAMPLE_IDS)]
print(f"Hours: {list(HOURS)}; polygons: {SAMPLE_IDS}")
# As the weights flow does
polygons = polygons.to_crs("EPSG:4326")
polygons.geometry = polygons.geometry.make_valid(method="structure", keep_collapsed=False)
polygons[["id", "name"]] if "name" in polygons else polygons[["id"]]

In [ ]:
def open_aorc_year(year: int, variable: str) -> xr.DataArray:
    """As aorc_mean_areal_values.open_aorc_year: lazily opened, decoded to float32, with its CRS."""
    store = S3Store(
        "noaa-nws-aorc-v1-1-1km", prefix=f"{year}.zarr", region="us-east-1", skip_signature=True
    )
    ds = xr.open_zarr(
        ObjectStore(store, read_only=True), zarr_format=2, consolidated=True, chunks=None, decode_cf=False
    )[[variable]]
    for name in ("scale_factor", "add_offset"):
        if name in ds[variable].attrs:
            ds[variable].attrs[name] = np.float32(ds[variable].attrs[name])
    return (
        xr.decode_cf(ds)[variable]
        .rio.write_crs("EPSG:4326")
        .rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
    )


assert HOURS.year.nunique() == 1, "Keep HOURS within one year (one AORC store)."
da = open_aorc_year(HOURS[0].year, GRID_VARIABLE_NAME)
minx, miny, maxx, maxy = polygons.total_bounds
grid = (
    da.sel(time=HOURS)
    .rio.clip_box(minx - 0.05, miny - 0.05, maxx + 0.05, maxy + 0.05)
    .sortby("latitude", ascending=False)  # north-up for the GeoTIFF
    .load()
)
# Drop the source CF encoding so the GeoTIFFs get the decoded float32 values
grid.encoding = {}
grid.attrs = {
    k: v for k, v in grid.attrs.items() if k not in ("_FillValue", "missing_value", "scale_factor", "add_offset")
}
grid.rio.write_nodata(np.nan, encoded=False, inplace=True)
grid.sizes

In [ ]:
rows = []
with tempfile.TemporaryDirectory() as tmp:
    # Relative pixel areas on the lat/lon grid
    area_path = Path(tmp) / "area.tif"
    (np.cos(np.deg2rad(grid.latitude)) * xr.ones_like(grid.isel(time=0))).rio.to_raster(area_path)
    for t in HOURS:
        path = Path(tmp) / "hour.tif"
        grid.sel(time=t).rio.to_raster(path)
        # exactextract replaces an "id" column with row numbers, so pass the ids as location_id
        ee = exact_extract(
            str(path), polygons.assign(location_id=polygons["id"]), "weighted_mean", weights=str(area_path),
            include_cols=["location_id"], output="pandas",
        ).set_index("location_id")["weighted_mean"]
        for _, poly in polygons.iterrows():
            centers = grid.sel(time=t).rio.clip([poly.geometry], all_touched=False, drop=True)
            rows.append({
                "location_id": poly["id"],
                "value_time": t,
                "exactextract": float(ee[poly["id"]]) * VALUE_SCALE,
                "pixel_centers": float(centers.mean(skipna=True)) * VALUE_SCALE,
            })
independent = pd.DataFrame(rows)

In [ ]:
warehouse = ev.spark.sql(f"""
    SELECT location_id, value_time, value AS warehouse
    FROM {table}
    WHERE {where}
      AND location_id IN ({", ".join(f"'{i}'" for i in polygons["id"])})
      AND value_time IN ({", ".join(f"TIMESTAMP '{t}'" for t in HOURS)})
""").toPandas()

check = independent.merge(warehouse, on=["location_id", "value_time"], how="left")
check["matches_exactextract"] = np.isclose(check["warehouse"], check["exactextract"], rtol=1e-5, atol=1e-6 * VALUE_SCALE)
check["pct_diff_pixel_centers"] = 100 * (check["warehouse"] - check["pixel_centers"]) / check["pixel_centers"]
check

In [ ]:
assert check["warehouse"].notna().all(), "Warehouse is missing rows for some sampled polygon-hours."
assert check["matches_exactextract"].all(), "Warehouse values differ from exactextract."
print(f"All {len(check)} sampled values match exactextract.")

## 3. Rerun check

Snapshot the values, rerun the flow over the same `START..END` (e.g. in local mode after an executor run), then
compare. The values should be identical and the row count unchanged.

In [ ]:
before = spark.sql(f"SELECT location_id, value_time, value FROM {table} WHERE {where}").toPandas()
len(before)

Rerun the flow now, then run the next cell.

In [ ]:
after = spark.sql(f"SELECT location_id, value_time, value FROM {table} WHERE {where}").toPandas()
keys = ["location_id", "value_time"]
pd.testing.assert_frame_equal(
    before.sort_values(keys).reset_index(drop=True),
    after.sort_values(keys).reset_index(drop=True),
    check_exact=True,
)
print(f"Rerun left {len(after)} rows, identical to before.")